# Ward Hierarchical Clustering

**DS4DH · Module 06 — Clustering and Segmentation**

*Technique:* Ward linkage, dendrograms, and cross-algorithm agreement as validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/06c_ward_hierarchical.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

K-Means starts from k and partitions. Hierarchical clustering starts from every
point as its own cluster and repeatedly merges the two whose union increases
within-cluster variance least — that rule is **Ward linkage**.

You do not choose k in advance. You build the whole tree and cut it wherever you
like, which makes the choice of k visible rather than assumed.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
RANDOM_STATE = 42
N_CLUSTERS = 4

X = StandardScaler().fit_transform(feat[FEATURES])
Z = linkage(X, method='ward')

print(f'linkage matrix: {Z.shape[0]} merges for {len(feat)} CSDs')
print()
print('The last few merges — height is the cost of joining:')
print(f'{"merge":>7}{"height":>12}')
for i, row in enumerate(Z[-6:], start=Z.shape[0] - 6):
    print(f'{i:>7}{row[2]:>12.2f}')
print()
print('A large jump in height means you merged two groups that were genuinely')
print('far apart — that gap is where the tree wants to be cut.')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, ax=ax, no_labels=True, color_threshold=Z[-(N_CLUSTERS - 1), 2])
ax.axhline(Z[-(N_CLUSTERS - 1), 2], color='#E8663D', ls='--',
           label=f'cut for {N_CLUSTERS} clusters')
ax.set_ylabel('Ward linkage distance')
ax.set_title('Hierarchical structure of 155 CSDs')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ward_labels = fcluster(Z, N_CLUSTERS, criterion='maxclust')
km_labels = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
                   n_init=10).fit_predict(X)

feat = feat.assign(ward=ward_labels, kmeans=km_labels)

print('Ward cluster sizes:  ', np.bincount(ward_labels)[1:])
print('K-Means cluster sizes:', np.bincount(km_labels))

## Cross-algorithm agreement

Two methods with different assumptions finding the same groups is evidence the
groups are in the data rather than in the method. Because the label *numbers* are
arbitrary, agreement has to be measured after matching each K-Means cluster to
its most overlapping Ward cluster.

In [ ]:
ct = pd.crosstab(feat['kmeans'], feat['ward'])
print(ct.to_string())
print()

matched = ct.values.max(axis=1).sum()
agreement = matched / len(feat)
print(f'best-match agreement: {matched} of {len(feat)} CSDs = {agreement:.1%}')
print()
print('High agreement means the typology is a property of the data. The CSDs')
print('the two methods disagree about are the genuine border cases.')

In [ ]:
# A label-invariant measure of the same thing.
from sklearn.metrics import adjusted_rand_score
print(f'adjusted Rand index: {adjusted_rand_score(feat["kmeans"], feat["ward"]):.3f}')
print('  1.0 = identical partitions, 0.0 = no better than chance')

In [ ]:
# Who are the border cases? These are the places worth looking at by hand.
best_match = ct.values.argmax(axis=1)
disagree = feat[[ct.columns[best_match[k]] != w
                 for k, w in zip(feat['kmeans'], feat['ward'])]]

print(f'{len(disagree)} CSDs assigned differently by the two methods:')
print()
cols = ['geography_name', 'cma', 'Total', 'tot_income', 'kmeans', 'ward']
print(disagree[cols].head(12).to_string(index=False))

### 🔧 Your turn 1

Change `method='ward'` to `'complete'`, then `'average'`, re-running the
dendrogram and the agreement.

Which linkage agrees best with K-Means, and why would you expect that? (Ward
minimises within-cluster variance, which is also what K-Means optimises.)

### 🔧 Your turn 2

Change `N_CLUSTERS` to 3 and 5 and record the agreement each time.

Does agreement peak at a particular k? A k where two different algorithms agree
most is a reasonable, data-driven argument for that k.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Ward agrees best with K-Means, and the reason is structural:
both minimise within-cluster variance, so they are optimising nearly the same
objective by different search strategies. Complete linkage produces more compact,
equal-sized clusters and agrees less; average linkage falls between. When you
report cross-algorithm agreement as validation, you should use a method that does
*not* share your first method's objective — otherwise the agreement is partly
built in. Ward-versus-K-Means agreement is therefore weaker evidence than it
looks, and complete-linkage agreement is stronger evidence.

**Your turn 2.** Agreement is high across k = 3–5, typically peaking around 3–4.
That stability is a better argument for the chosen k than the silhouette score
alone, because it does not depend on one algorithm's geometry. Combine it with
the "can I name each cluster in a sentence" test from 06b and you have a defensible
choice you can explain to a non-technical reader.

</details>

## Where this stops

You have a typology validated two ways. It is descriptive: it says these places
resemble each other, not why, and not what would change if policy changed.

Next: Module 07 swaps interpretability for predictive accuracy, then tries to win
some interpretability back.